# 03 - Prefix acceptance, SOL, concurrency

**학습 목표**: prefix-only verify가 비연속 정답 token을 버리는 정도와, concurrency에 따라 mode 선택이 바뀔 수 있음을 toy model로 확인합니다.

**실행 방법**: Python 3/Jupyter에서 셀을 위에서 아래 순서로 실행합니다. 외부 패키지는 필요하지 않으며 Python 표준 라이브러리만 사용합니다.

In [ ]:
import random
import statistics

def longest_prefix(matches):
    count = 0
    for match in matches:
        if not match:
            break
        count += 1
    return count

rng = random.Random(23)
prefix_counts, nonprefix_counts = [], []
for _ in range(5000):
    matches = [rng.random() < 0.82 for _ in range(32)]
    prefix_counts.append(longest_prefix(matches))
    nonprefix_counts.append(sum(matches))

print('mean prefix-safe matches:', round(statistics.mean(prefix_counts), 2))
print('mean all matching positions:', round(statistics.mean(nonprefix_counts), 2))
assert statistics.mean(nonprefix_counts) > statistics.mean(prefix_counts)

In [ ]:
def toy_per_user_throughput(mode, concurrency):
    # AR은 batching 이득이 크고, self-spec은 low concurrency에서 memory read를 절약한다고 가정합니다.
    if mode == 'AR':
        system = 100 * min(concurrency, 32) ** 0.72
    elif mode == 'linear-SS':
        system = 260 * min(concurrency, 12) ** 0.45
    else:
        system = 210 * min(concurrency, 8) ** 0.35
    return system / concurrency

for c in (1, 2, 8, 32, 128):
    values = {m: toy_per_user_throughput(m, c) for m in ('AR', 'linear-SS', 'diffusion')}
    best = max(values, key=values.get)
    print(f'concurrency={c:3d} best={best:9s}', {k: round(v, 1) for k, v in values.items()})

SOL은 `all matching positions`를 그대로 commit하는 방식이 아닙니다. commit 후에도 serial diffusion target이 유지되는지 비싼 안전성 탐색을 합니다. 위 첫 실험은 prefix-only 손실의 방향만 보여줍니다. 두 번째 식도 논문 GPU 측정치가 아닌 설명용 곡선입니다.